In [33]:
from GradientGang.TheFreePirate.DataHandling.PirateDataModule import PirateDataModule
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
allJoints = [f'joint_{i:02d}' for i in range(0, 30)]
# allJoints.remove('joint_13')  # Remove joint_13 as it is not useful for pain prediction
# allJoints.remove('joint_17')  # Remove joint_13 as it is not useful for pain prediction

# print(allJoints)

dataParams = {
    'folderPath': '../../dataset/PirateProcessed',
    'trainTimeSeriesFileName': 'pirate_pain_train.csv',
    'trainGlobalFeaturesFileName': 'train_global_features.csv',
    'trainLabelsFileName': 'pirate_pain_train_labels.csv',
    'testTimeSeriesFileName': 'pirate_pain_test.csv',
    'testGlobalFeaturesFileName': 'test_global_features.csv',
    'labelsMapping': {'no_pain': 0, 'low_pain': 1, 'high_pain': 2},
    'batch_size': 64,
    'num_workers': 0,
    'columnsToIgnore': ['pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4', 'joint_25', 'joint_26', 'joint_00', 'joint_02', 'joint_03', 'joint_05', 'joint_06'],
}

dataModule = PirateDataModule.fromCSV(**dataParams)

In [35]:
from GradientGang.TheFreePirate.DataHandling.DataAugmentation.TensorAugmenter import SequentialAugmenter, ScaleAugmenter, JitterAugmenter, WindowingAugmenter, TimeWarpAugmenter, OffsetAugmenter

dataAugmenter = SequentialAugmenter(
        augmenters=[
            ScaleAugmenter(rangeScale=0.1),
            JitterAugmenter(jitterStdDev=0.1),
            OffsetAugmenter(rangeOffset=0.1),
            TimeWarpAugmenter(maxWarpFraction=0.1),
        ],
        nCopies=0,
        keepOriginal=True
    )

windowingAugmenter = WindowingAugmenter(windowSize=10, stride=5)

dataModule.setupKFolds(nFolds=5, dataAugmenter=dataAugmenter, windowingAugmenter=windowingAugmenter, augmentTestSet=True)

trainLoader, valLoader = dataModule.getFoldDataLoaders(foldIndex=0)

#print (trainLoader, valLoader)
for batch in trainLoader:
    (timeSeriesBatch, globalFeaturesBatch), labelsBatch = batch
    print("Time Series Batch Shape:", timeSeriesBatch.shape)
    print("Global Features Batch Shape:", globalFeaturesBatch.shape)
    print("Labels Batch Shape:", labelsBatch.shape)
    break

for batch in valLoader:
    (timeSeriesBatch, globalFeaturesBatch), labelsBatch = batch
    print("Time Series Batch Shape:", timeSeriesBatch.shape)
    print("Global Features Batch Shape:", globalFeaturesBatch.shape)
    print("Labels Batch Shape:", labelsBatch.shape)
    break

[PirateDataModule] Setting up K-Folds(5) with data augmentation...
[PirateDataModule] K-Folds setup completed with 5 folds.
Time Series Batch Shape: torch.Size([64, 31, 10, 23])
Global Features Batch Shape: torch.Size([64, 33])
Labels Batch Shape: torch.Size([64])
Time Series Batch Shape: torch.Size([64, 31, 10, 23])
Global Features Batch Shape: torch.Size([64, 33])
Labels Batch Shape: torch.Size([64])


In [36]:
from GradientGang.TheFreePirate.Architectures.PirateLightningModule import PirateLightningModule
from pytorch_lightning import Trainer

nFeats = 23

architectureParams = {
    'f1AverageStrategy': 'weighted',
    'numClasses': 3,
    'reconstructionLossWeight': 0.5,
    'useGlobalFeatures': False,
    'learningRate': 1e-3,
    'weightDecay': 1e-2,
    'activationFunction': 'relu',
    'dataModuleInfo': dataModule.getDataInfoKFold(),
    'timeSeriesEncoderNumLayers': 2,
    'timeSeriesEmbeddingDim': 64,
    'timeSeriesDropout': 0.5,
    'predictorNumLayers': 2,
    'predictorDropout': 0.1,
    }

model = PirateLightningModule(params=architectureParams)

trainer = Trainer(max_epochs=100, accelerator='cpu', devices=1)

(trainDataLoader, valDataLoader) = dataModule.getFoldDataLoaders(foldIndex=0)
trainer.fit(model, train_dataloaders=trainDataLoader, val_dataloaders=valDataLoader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name               | Type              | Params | Mode 
-----------------------------------------------------------------
0 | classificationLoss | CrossEntropyLoss  | 0      | train
1 | reconstructionLoss | MSELoss           | 0      | train
2 | f1                 | MulticlassF1Score | 0      | train
3 | timeSeriesEncoder  | FeedForwardModel  | 68.7 K | train
4 | classifier         | FeedForwardModel  | 3.9 K  | train
5 | timeSeriesDecoder  | FeedForwardModel  | 68.9 K | train
-----------------------------------------------------------------
141 K     Trainable params
0         Non-trainable params
141 K     Total params
0.566     Total estimated model params size (MB)
30        Modules in train mode
0    

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Polimi\Master\3sem\ANN_challenges\GradientGang\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Polimi\Master\3sem\ANN_challenges\GradientGang\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Polimi\Master\3sem\ANN_challenges\GradientGang\.venv\Lib\site-packages\pytorch_lightning\loops\fit_loop.py:310: The number of training batches (29) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

c:\Polimi\Master\3sem\ANN_challenges\GradientGang\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
